In [ ]:

 
from src.api import getHistoricoMOW, getInfoEstacionesComerciales
from src.api.api import GraylogAPIProcessor
from src.api.APIs import getInfoFiabilidadEstacion
from src.processor import MIEProcessor, XSIVProcessor, XPECProcessor  # LogProcessor,
from src.processor.log_procesor import LogProcessor
from src.utils import formatTimedelta  # loadViasFromTopos,
from io import StringIO
from src.utils import (
    dateFromText,
    getEstacionamientos,
    getFilesByDate,
    getFilesByWeek,
    guardarExcel,
    guardarExcelMulti,
    isEmpty,
    isValidCode,
    listTopos,
    loadEstaciones,
    localizeFecha,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    rellenarId,
    setEF,
    slidingWindow,
    splitDataframe,
    time2localtime,
    splitList,
)
 
# from src.visualizacion.visualizaciones import visualizacionOcupacionVia
from src.visualizacion.visualizaciones import getSortedPlatforms
from src.api.APIs import hacerPeticion
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex

import gzip
import shutil


In [ ]:
dir_logs = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20251105033523.xml.gz")
dir_dest_logs = Path (r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20251105033523.xml")

In [ ]:

with gzip.open(dir_logs, 'rb') as f_in:
    with open(dir_dest_logs, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Archivo descomprimido correctamente.")


In [ ]:
xpec= XPECProcessor()

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\xPEC\xPEC_20251029033553.xml")

In [ ]:
data= xpec.loadLogFile(fname)

In [ ]:
tren  = data[data["NTécnico"] == "51605"]

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Desarrollo\Info SITRA\xpec_51605.xlsx")

In [ ]:
guardarExcel(tren,fname)

In [ ]:
dir_logs = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra\WO0000000537852 xMSG_MSECentral")

for gz_file in dir_logs.glob("*.gz"):
    csv_file = gz_file.with_suffix('.csv')
    with gzip.open(gz_file, 'rb') as f_in:
        with open(csv_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print(f"Descomprimido: {gz_file.name} -> {csv_file.name}")

In [ ]:
fnames = []
for fname in dir_logs.rglob("*.*"):
    if fname.is_file() and  fname.suffix == ".csv":
        fnames.append(fname)



In [ ]:
from src.processor import  SitraProcessor

In [ ]:
logs = []
for fname in fnames:
    try:
        with fname.open("r", encoding="utf-8") as f:  # Asegúrate de usar el encoding correcto
            data = f.read()
            data = regex.sub(r'^"timestamp","message"\n?', "", data)
            data = regex.sub(r" +", "", data)
            if regex.search(r'("{4}|""[\w\d\.-]+?"")', data):
                data = regex.sub(r'""', '"', data)
                lines = data.split("\n")
                logs.extend([
                    regex.split(r"<\?xml.+?\?>", l.strip('"'))[-1]
                    for l in lines if l.strip()
                ])
    except Exception as e:
        print(f"Error al procesar el archivo {fname}: {e}")


In [ ]:
logs = []
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Sitra\WO0000000528043\xMSG_MSECentral.log.11.59.csv")
with fname.open("r") as f:
        data = f.read()
        data = regex.sub(r'^"timestamp","message"\n?', "", data)
        data = regex.sub(r" +", "", data)
        if regex.search(r'("{4}|""[\w\d\.-]+?"")', data):
                data = regex.sub(r'""', '"', data)
        lines = data.split("\n")
        logs = [
            regex.split(r"<\?xml.+?\?>", l.strip('"'))[-1]
            # l.split('<?xml version="1.0" encoding="UTF-8" standalone="yes"?>')[-1]
            for l in lines
        ]
    


In [ ]:
ruOperation = [el for el in logs if "ruOperationRequest" in el]
xmls = f"<xml>{''.join(ruOperation)}</xml>"
df_ru11 = pd.read_xml(StringIO(xmls))
#df_ru = df_ru[list(rename_cols.keys())].rename(columns=rename_cols)
df_ru11["runningNumber"] = df_ru11["runningNumber"].apply(rellenarId)
df_ru11.head(5)

In [ ]:
df_ru2

In [ ]:
path_g=Path(r"C:\Users\xiangzhou.zhang\Documents\CSV\ru_0105_1105.xlsx")

In [ ]:
data = {"01-05": df_ru1,
        "02-05": df_ru2,
        "03-05":df_ru3,
        "04-05":df_ru4,
        "05-05": df_ru5,
        "06-05": df_ru6,
        "07-05": df_ru7,
        "08-05": df_ru8,
        "09-05": df_ru9,
        "10-05":df_ru10,
        "11-05":df_ru11
        }
guardarExcelMulti(data,path_g)